In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import os
import asyncio

In [9]:
load_dotenv(override=True)

True

In [10]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [11]:
sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model="gpt-5-nano")
sales_agent2 = Agent(name="Engaging Sales Agent", instructions=instructions2, model="gpt-5-nano")
sales_agent3 = Agent(name="Busy Sales Agent", instructions=instructions3, model="gpt-5-nano")

In [12]:
result = Runner.run_streamed(sales_agent1, input="write a cold sales email")  # we're not using await here but in below line is async for
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)


Subject: Simplify SOC 2 readiness for {Company} with AI

Hi {FirstName},

I’m {YourName} from ComplAI. We help fast-growing teams prepare for SOC 2 audits with an AI-powered platform that:
- automates evidence collection and control mapping
- centralizes audit artifacts
- provides continuous compliance monitoring

This can shorten audit cycles and reduce last-minute auditor questions. I’d love to show you a 15-minute demo and discuss how it fits {Company}’s controls and timeline.

Are you available {Date/Time} or {Alternative}?

Best regards,
{YourName}
{Title}
ComplAI
{Phone}
{Email}
{Website}

P.S. If you’re not the right person, could you connect me with the SOC 2 owner?

In [13]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model="gpt-5-nano"
)

In [ ]:
message = "Write a cold sales email"
with trace("selecting the best email"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]
    emails = "cold emails: \n\n".join(outputs)
    best = await Runner.run(sales_picker, emails)

    print(f"best email: {best.final_output}")
